# Transformer model architecture - gold

Defines the model itself: input projection, positional encoding, stacked self-attention layers, and an output head. This notebook only builds and sanity-checks the architecture (confirms it runs and produces the right output shape) - training happens in the next notebook.

In [1]:
import math
import torch
import torch.nn as nn

## Positional encoding

Fixed sine/cosine pattern, one per day-position, added to that day's embedding. Not learned - computed once and reused every time.

In [2]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe)

    def forward(self, x):
        # x shape: (batch, window_size, d_model)
        return x + self.pe[: x.size(1)]

## The full model

- `input_projection`: 1 number -> 32-dimensional vector, per day
- `pos_encoding`: adds position information
- `encoder`: 2 stacked `TransformerEncoderLayer`s (each bundles multi-head attention + feed-forward + residual/norm), 4 attention heads, 10% dropout
- `output_head`: takes the *last* day's final vector (already attended to the whole window) -> 1 number
- `softplus`: guarantees the output is positive, since volatility can't be negative

In [3]:
class VolatilityTransformer(nn.Module):
    def __init__(self, d_model=32, num_heads=4, num_layers=2, dropout=0.1):
        super().__init__()
        self.input_projection = nn.Linear(1, d_model)
        self.pos_encoding = PositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=64,  # PyTorch defaults to 2048, wildly oversized for d_model=32 and ~2,900 training examples
            dropout=dropout,
            batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.output_head = nn.Linear(d_model, 1)

    def forward(self, x):
        # x shape: (batch, window_size, 1)
        x = self.input_projection(x)       # -> (batch, window_size, d_model)
        x = self.pos_encoding(x)
        x = self.encoder(x)                # -> (batch, window_size, d_model)
        last_day = x[:, -1, :]             # -> (batch, d_model)
        out = self.output_head(last_day)   # -> (batch, 1)
        return nn.functional.softplus(out).squeeze(-1)


model = VolatilityTransformer()
num_params = sum(p.numel() for p in model.parameters())
print(f"Total trainable parameters: {num_params:,}")
model

Total trainable parameters: 17,185


VolatilityTransformer(
  (input_projection): Linear(in_features=1, out_features=32, bias=True)
  (pos_encoding): PositionalEncoding()
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=32, out_features=32, bias=True)
        )
        (linear1): Linear(in_features=32, out_features=64, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=64, out_features=32, bias=True)
        (norm1): LayerNorm((32,), eps=1e-05, elementwise_affine=True, bias=True)
        (norm2): LayerNorm((32,), eps=1e-05, elementwise_affine=True, bias=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (output_head): Linear(in_features=32, out_features=1, bias=True)
)

## Sanity check: run one real batch through it

No training has happened yet, so these predictions are meaningless (the dials are still random) - this just confirms the architecture is wired correctly and produces the right output shape, and that every prediction is positive as designed.

In [4]:
import numpy as np
import pandas as pd
from torch.utils.data import TensorDataset, DataLoader

WINDOW_SIZE = 30

gold = pd.read_csv("../data/gold_futures.csv", skiprows=[1, 2], index_col=0, parse_dates=True)
returns = gold["Close"].pct_change().dropna() * 100


def create_windows(returns, window_size):
    values = returns.values
    X, y = [], []
    for start in range(len(values) - window_size):
        end = start + window_size
        X.append(values[start:end])
        y.append(abs(values[end]))
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)


X, y = create_windows(returns, WINDOW_SIZE)
X_tensor = torch.tensor(X).unsqueeze(-1)
y_tensor = torch.tensor(y)

sample_loader = DataLoader(TensorDataset(X_tensor, y_tensor), batch_size=8)
batch_X, batch_y = next(iter(sample_loader))

model.eval()
with torch.no_grad():
    predictions = model(batch_X)

print(f"Input shape:  {batch_X.shape}")
print(f"Output shape: {predictions.shape}")
print(f"Predictions (untrained, still random dials): {predictions}")
print(f"All positive? {(predictions > 0).all().item()}")

Input shape:  torch.Size([8, 30, 1])
Output shape: torch.Size([8])
Predictions (untrained, still random dials): tensor([0.6936, 0.7105, 0.6644, 0.7760, 0.7872, 0.7664, 0.5762, 0.5850])
All positive? True
